# Citation data

Note:

- The resulting data can be found in the folder `empirical_networks`. 


Clean up:

1. `create_author_network`: Should we delete the lines of code that were commented out?
2. Perceptron: The code for the perceptron is still in here; should we delete it?


In [1]:
import dill
import copy
import json
import numpy as np
import pandas as pd
import networkx as nx
import pickle

from itertools import chain, chain#, batched
from tqdm.auto import tqdm

from pyalex import Works, Authors, Sources, Institutions, Concepts, Publishers, Funders, config

config.email = "h.w.a.duijf@uu.nl"
config.max_retries = 5

/Users/ignacio/Documents/VS Code/GitHub Repositories/e_network_inequality/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Methods

### Prune citation data: only articles, and remove articles without bibliography

In [2]:
def get_works_with_references(works: list) -> list:
    works_pruned: list = []
    for work in works:
        try:
            assert work["referenced_works"] != []
            works_pruned.append(work)
        except:
            pass
    return works_pruned

In [3]:
def get_articles(works: list) -> list:
    articles: list = []
    for work in works:
        try:
            assert work["primary_location"]["source"]["type"] == "journal"
            assert work["type"] == "article"
            articles.append(work)
        except:
            pass
    return articles

### Create author network from dataframe of records

In [4]:
def create_author_network(works: list) -> nx.DiGraph:
    """Create a directed author citation network from a list of works.
    Arguments
    ---------
    works : list[Work]
        A list of works (see PyAlex).
    
    Returns
    -------
    G : nx.DiGraph
        A directed graph where nodes are authors and edges represent citations
        from cited authors to citing authors."""
    # Create a directed graph and dataframe of works
    df = pd.DataFrame(works)
    G: nx.DiGraph = nx.DiGraph()

    # Add nodes and edges
    for _, row in df.iterrows():
        for author in row['authorships']:
            this_author = author['author']['id'] 
            
            # ignore the author with the id "A9999999999" as it is a placeholder for missing values
            if (this_author is None) or (this_author.split("/")[-1] == "A9999999999"): 
                continue
            
            # add node if the author is not already present in the network
            if this_author not in G.nodes():
                G.add_node(this_author)
                G.nodes()[this_author]['n_works'] = 1
            else:
                G.nodes()[this_author]['n_works'] += 1
            
            # Add edges
            for cited_work_id in row["referenced_works"]:
                cited_work = df[df['id'] == cited_work_id] # This fails silently if citations are not present!
                if len(cited_work) >= 1: # In case of multiple hits (shouldn't happen once sampling is fixed)
                    cited_work = cited_work.iloc[0]
                
                for cited_author in cited_work['authorships']:
                    cited_author = cited_author['author']['id'] 
                    
                    if cited_author not in G.nodes():
                        if (cited_author is not None) and (cited_author.split("/")[-1] != "A9999999999"): 
                            G.add_node(cited_author)
                            G.nodes()[cited_author]['n_works'] = 0
                    
                    # edges go FROM cited TO citing
                    if cited_author in G.nodes() and not G.has_edge(cited_author, this_author):
                        G.add_edge(cited_author, this_author)

    return G

### Pruning by removing ‘twins’ (aka, strong co-authors)

In [5]:
def generate_twins_dict(net: nx.DiGraph, records: list) -> dict:
    
    authors_twins_dict: dict = {}

    for author_id in tqdm(net.nodes()):
        author_records = [
            work for work in records 
            if author_id in [author["author"]["id"] for author in work["authorships"]]]
        
        twins = set()
        for k, record in enumerate(author_records):
            if k == 0:
                coauthors = [
                    coauthor["author"]["id"] 
                    for coauthor in record["authorships"]
                    if coauthor["author"]["id"] != author_id
                ]
                twins = set(coauthors)
            elif twins == set():
                break
            else:
                coauthors = [
                    coauthor["author"]["id"] 
                    for coauthor in record["authorships"]
                    if coauthor["author"]["id"] != author_id
                ]
                twins = twins.intersection(set(coauthors))
        if twins:
            authors_twins_dict[author_id] = twins
    return authors_twins_dict

In [6]:
def prune_network(net: nx.DiGraph, authors_twins_dict: dict) -> nx.DiGraph:
    network_pruned = copy.deepcopy(net)
    for author_id, twins in tqdm(authors_twins_dict.items()):
        twins_in_network = [twin for twin in twins if twin in network_pruned.nodes()]
        if twins_in_network:
            network_pruned.remove_node(author_id)
    return network_pruned

### Pruning by taking the largest weakly connected component

In [7]:
def produce_lcc(net: nx.DiGraph) -> nx.DiGraph:
    # Extract largest component:
    largest_cc = max(nx.weakly_connected_components(net), key=len)
    
    lcc = nx.DiGraph()
    lcc.add_nodes_from((n, net.nodes[n]) for n in largest_cc)
    lcc.add_edges_from((n, nbr, d)
        for n, nbrs in net.adj.items() if n in largest_cc
        for nbr, d in nbrs.items() if nbr in largest_cc)
    lcc.graph.update(net.graph)
    # lcc = copy.deepcopy(net.subgraph(largest_cc))
    return lcc

In [8]:
def remove_self_loops(net: nx.DiGraph) -> nx.DiGraph:
    network_pruned = copy.deepcopy(net).copy()
    for node in net.nodes():
        if (node, node) in net.edges():
            network_pruned.remove_edge(node, node)
    return network_pruned

## Citation data from OpenAlex

In [9]:
def set_version(self, v):
#     self._add_params("data-version", str(v))
    return self

Works.version = set_version

# results = Works().filter(publication_year=2020).version(2).get()

def OA_full_text_search(text: str, year: str, version: int=1) -> list:
    """Note: version=1 takes the old OA, version=2 takes the new OA Waldren"""
    query = Works().search(f'"{text}"').filter(publication_year=year)
    works: list = []

    for _, work in enumerate(chain(*query.paginate(per_page=200, n_max=None))):
        works.append(work)
    return works

def OA_title_abstract_search(text: str, year: str, version: int=1) -> list:
    """Note: version=1 takes the old OA, version=2 takes the new OA Waldren"""
    query = Works().search_filter(title_and_abstract=f'"{text}"').filter(publication_year=year)
    works: list = []

    for _, work in enumerate(chain(*query.paginate(per_page=200, n_max=None))):
        works.append(work)
    return works

### Peptic ulcer disease

Get the records

In [10]:
# Title and abstract search using OpenAlex version 1
works_title_1 = OA_title_abstract_search(text="peptic ulcer disease", year="1900-1978", version=1) 
print(f"{len(works_title_1)=:,}")
articles_title_1 = get_articles(works_title_1)
print(f"{len(articles_title_1)=:,}")

len(works_title_1)=488
len(articles_title_1)=355


In [11]:
# Title and abstract search using OpenAlex version 2
works_title_2 = OA_title_abstract_search(text="peptic ulcer disease", year="1900-1978", version=2) 
print(f"{len(works_title_2)=:,}")
articles_title_2 = get_articles(works_title_2)
print(f"{len(articles_title_2)=:,}")

len(works_title_2)=488
len(articles_title_2)=355


In [12]:
# Full text search using OpenAlex version 2
works_full_2 = OA_full_text_search(text="peptic ulcer disease", year="1900-1978", version=2)
print(f"{len(works_full_2)=:,}")
articles_full_2 = get_articles(works_full_2)
print(f"{len(articles_full_2)=:,}")


len(works_full_2)=683
len(articles_full_2)=522


In [13]:
# Full text search using OpenAlex version 1
works_pud = OA_full_text_search(text="peptic ulcer disease", year="1900-1978", version=1)
print(f"{len(works_pud)=:,}") 


len(works_pud)=683


In [14]:
with open('pud_works.pkl', 'wb') as f:
    dill.dump(works_pud, f)

In [15]:
with open('pud_works.pkl', 'rb') as f:
    works_pud = dill.load(f)

In [16]:
works_pud_pruned = get_works_with_references(works_pud)
print(f"{len(works_pud_pruned)=:,}")
articles_pud = get_articles(works_pud_pruned)
print(f"{len(articles_pud)=:,}")

len(works_pud_pruned)=464
len(articles_pud)=407


Create author-based network

In [17]:
network_pud_original = create_author_network(articles_pud) 
print(f"{network_pud_original.number_of_nodes()=:,}")
print(f"{network_pud_original.number_of_edges()=:,}")

network_pud_original.number_of_nodes()=974
network_pud_original.number_of_edges()=1,461


Prune author-based network

In [18]:
authors_twins_dict = generate_twins_dict(network_pud_original, works_pud)
network_pud_pruned = prune_network(network_pud_original, authors_twins_dict)
print(f"{network_pud_pruned.number_of_nodes()=:,}")
print(f"{network_pud_pruned.number_of_edges()=:,}")

network_pud_pruned_lcc = produce_lcc(network_pud_pruned)
print(f"{network_pud_pruned_lcc.number_of_nodes()=:,}")
print(f"{network_pud_pruned_lcc.number_of_edges()=:,}")

network_pud_final = remove_self_loops(network_pud_pruned_lcc)
print(f"{network_pud_final.number_of_nodes()=:,}")
print(f"{network_pud_final.number_of_edges()=:,}")

100%|██████████| 831/831 [00:00<00:00, 328322.03it/s]

network_pud_pruned.number_of_nodes()=337
network_pud_pruned.number_of_edges()=224
network_pud_pruned_lcc.number_of_nodes()=87
network_pud_pruned_lcc.number_of_edges()=182
network_pud_final.number_of_nodes()=87
network_pud_final.number_of_edges()=160


In [19]:
info_dict = {
    "data_type": 
        ["works", 
        "works with refs", 
        "articles", 
        "author network",
        "author network pruned",
        "author network pruned lcc",
        "author network final"],
    "number_of_nodes": [
        f"{len(works_pud):,.0f}", 
        f"{len(works_pud_pruned):,.0f}", 
        f"{len(articles_pud):,.0f}", 
        f"{network_pud_original.number_of_nodes():,.0f}", 
        f"{network_pud_pruned.number_of_nodes():,.0f}", 
        f"{network_pud_pruned_lcc.number_of_nodes():,.0f}", 
        f"{network_pud_final.number_of_nodes():,.0f}"
    ],
    "number_of_edges": [
        "N/A",  # works do not have edges
        "N/A",  # works with refs do not have edges
        "N/A",  # articles do not have edges
        f"{network_pud_original.number_of_edges():,.0f}", 
        f"{network_pud_pruned.number_of_edges():,.0f}", 
        f"{network_pud_pruned_lcc.number_of_edges():,.0f}", 
        f"{network_pud_final.number_of_edges():,.0f}"
    ]
}
df_info = pd.DataFrame(info_dict)
# df_info.astype({"number_of_edges": "Int64"})
df_info

,data_type,number_of_nodes,number_of_edges
0,works,683,N/A
1,works with refs,464,N/A
2,articles,407,N/A
3,author network,974,"1,461"
4,author network pruned,337,224
5,author network pruned lcc,87,182
6,author network final,87,160


Save networks

In [20]:
# with open('data/pud_works.pkl', 'w') as f:
#     dill.dump(works_pud, f)

# with open('data/pud_original.pkl', 'wb') as f:
#     dill.dump(network_pud_original, f)

with open('pud_final.pkl', 'wb') as f:
    dill.dump(network_pud_final, f)

# Save as JSON
from networkx.readwrite import json_graph
data_pud = json_graph.node_link_data(network_pud_final)
with open('pud_final.json', 'w') as f:
    json.dump(data_pud, f)

/Users/ignacio/Documents/VS Code/GitHub Repositories/e_network_inequality/.venv/lib/python3.10/site-packages/networkx/readwrite/json_graph/node_link.py:142: FutureWarning: 
The default value will be `edges="edges" in NetworkX 3.6.

To make this warning go away, explicitly set the edges kwarg, e.g.:

  nx.node_link_data(G, edges="links") to preserve current behavior, or
  nx.node_link_data(G, edges="edges") for forward compatibility.
  warnings.warn(


Loading the network from file

In [21]:
with open('pud_final.pkl', 'rb') as f:
    network = dill.load(f)

### Alternative PUD network

In [22]:
with open('pud_works.pkl', 'rb') as f:
    works_pud = dill.load(f)

In [23]:
works_pud_pruned = get_works_with_references(works_pud)
print(f"{len(works_pud_pruned)=:,}")
articles_pud = get_articles(works_pud_pruned)
print(f"{len(articles_pud)=:,}")

len(works_pud_pruned)=464
len(articles_pud)=407


In [24]:
network_pud_original = create_author_network(articles_pud) 
print(f"{network_pud_original.number_of_nodes()=:,}")
print(f"{network_pud_original.number_of_edges()=:,}")

network_pud_original.number_of_nodes()=974
network_pud_original.number_of_edges()=1,461


In [25]:
authors_twins_dict = generate_twins_dict(network_pud_original, works_pud)
network_pud_pruned = prune_network(network_pud_original, authors_twins_dict)
print(f"{network_pud_pruned.number_of_nodes()=:,}")
print(f"{network_pud_pruned.number_of_edges()=:,}")

network_pud_no_loops = remove_self_loops(network_pud_pruned)
print(f"{network_pud_no_loops.number_of_nodes()=:,}")
print(f"{network_pud_no_loops.number_of_edges()=:,}")

100%|██████████| 831/831 [00:00<00:00, 255233.35it/s]

network_pud_pruned.number_of_nodes()=337
network_pud_pruned.number_of_edges()=224
network_pud_no_loops.number_of_nodes()=337
network_pud_no_loops.number_of_edges()=196


In [26]:
import copy

def remove_zero_indegree_nodes(network: nx.DiGraph) -> nx.DiGraph:
    net_result = copy.deepcopy(network)
    for k in range(10_000):
        nodes_to_remove = [
            node for node in net_result.nodes() 
            if net_result.in_degree(node) == 0
        ]
        net_result.remove_nodes_from(nodes_to_remove)
        if nodes_to_remove == []:
            break
    return net_result

In [27]:
network_pud_nonzero_indegree = remove_zero_indegree_nodes(network_pud_no_loops)
print(f"{network_pud_nonzero_indegree.number_of_nodes()=:,}")
print(f"{network_pud_nonzero_indegree.number_of_edges()=:,}")

network_pud_nonzero_indegree.number_of_nodes()=32
network_pud_nonzero_indegree.number_of_edges()=65


In [28]:
network_lcc = produce_lcc(network_pud_nonzero_indegree)
print(f"{network_lcc.number_of_nodes()=:,}")
print(f"{network_lcc.number_of_edges()=:,}")

network_lcc.number_of_nodes()=32
network_lcc.number_of_edges()=65


In [29]:
with open('pud_alternative.pkl', 'wb') as f:
    dill.dump(network_lcc, f)

### Perceptron

In [30]:
string = "perceptron"
works_perceptron = OA_full_text_search(text=string, year="1900-1971") 

In [31]:
works_perceptron_pruned = get_works_with_references(works_perceptron)
print(f"{len(works_perceptron_pruned)=:,}")

len(works_perceptron_pruned)=101


In [32]:
articles_perceptron = get_articles(works_perceptron_pruned)
print(f"{len(articles_perceptron)=:,}")

len(articles_perceptron)=74


Create author-based network

In [33]:
network_perceptron_original = create_author_network(articles_perceptron) 
print(f"{network_perceptron_original.number_of_nodes()=:,}")
print(f"{network_perceptron_original.number_of_edges()=:,}")

network_perceptron_original.number_of_nodes()=116
network_perceptron_original.number_of_edges()=49


Prune author-based network

In [34]:
authors_twins_dict = generate_twins_dict(network_perceptron_original, articles_perceptron)
network_perceptron_pruned = prune_network(network_perceptron_original, authors_twins_dict)
print(f"{network_perceptron_pruned.number_of_nodes()=:,}")
print(f"{network_perceptron_pruned.number_of_edges()=:,}")

network_perceptron_pruned_lcc = produce_lcc(network_perceptron_pruned)
print(f"{network_perceptron_pruned_lcc.number_of_nodes()=:,}")
print(f"{network_perceptron_pruned_lcc.number_of_edges()=:,}")

network_perceptron_final = remove_self_loops(network_perceptron_pruned_lcc)
print(f"{network_perceptron_final.number_of_nodes()=:,}")
print(f"{network_perceptron_final.number_of_edges()=:,}")

100%|██████████| 79/79 [00:00<00:00, 181164.58it/s]

network_perceptron_pruned.number_of_nodes()=64
network_perceptron_pruned.number_of_edges()=19
network_perceptron_pruned_lcc.number_of_nodes()=10
network_perceptron_pruned_lcc.number_of_edges()=12
network_perceptron_final.number_of_nodes()=10
network_perceptron_final.number_of_edges()=9


Save networks

In [35]:
# with open('data/perceptron_works.pkl', 'wb') as f:
#     dill.dump(works_perceptron, f)

# with open('data/perceptron_original.pkl', 'wb') as f:
#     dill.dump(network_perceptron_original, f)
    
with open('perceptron_final_dill.pkl', 'wb') as f:
    dill.dump(network_perceptron_final, f)

# Save the object to a file
with open('perceptron_final.pkl', 'wb') as f:
    pickle.dump(network_perceptron_final, f)

# Save as JSON
data_perceptron = json_graph.node_link_data(network_perceptron_final)
with open('perceptron_final.json', 'w') as f:
    json.dump(data_perceptron, f)

In [36]:
info_dict = {
    "data_type": 
        ["works", 
        "works with refs", 
        "articles", 
        "author network",
        "author network pruned",
        "author network pruned lcc",
        "author network final"],
    "number_of_nodes": [
        f"{len(works_perceptron):,.0f}", 
        f"{len(works_perceptron_pruned):,.0f}", 
        f"{len(articles_perceptron):,.0f}", 
        f"{network_perceptron_original.number_of_nodes():,.0f}", 
        f"{network_perceptron_pruned.number_of_nodes():,.0f}", 
        f"{network_perceptron_pruned_lcc.number_of_nodes():,.0f}", 
        f"{network_perceptron_final.number_of_nodes():,.0f}"
    ],
    "number_of_edges": [
        "N/A",  # works do not have edges
        "N/A",  # works with refs do not have edges
        "N/A",  # articles do not have edges
        f"{network_perceptron_original.number_of_edges():,.0f}", 
        f"{network_perceptron_pruned.number_of_edges():,.0f}", 
        f"{network_perceptron_pruned_lcc.number_of_edges():,.0f}", 
        f"{network_perceptron_final.number_of_edges():,.0f}"
    ]
}
df_info = pd.DataFrame(info_dict)
# df_info.astype({"number_of_edges": "Int64"})
df_info

,data_type,number_of_nodes,number_of_edges
0,works,195,N/A
1,works with refs,101,N/A
2,articles,74,N/A
3,author network,116,49
4,author network pruned,64,19
5,author network pruned lcc,10,12
6,author network final,10,9


### Perceptron

In [37]:
# Use the new API function OA_full_text_search (version 1 to match old behavior or default logic)
works_perceptron = OA_full_text_search(text="perceptron", year="1900-2000", version=1)
print(f"{len(works_perceptron)=:,}")

len(works_perceptron)=6,212


In [38]:
works_perceptron_pruned = get_works_with_references(works_perceptron)
print(f"{len(works_perceptron_pruned)=:,}")

len(works_perceptron_pruned)=3,933


In [39]:
articles_perceptron = get_articles(works_perceptron_pruned)
print(f"{len(articles_perceptron)=:,}")

len(articles_perceptron)=2,386


Create author-based network

In [40]:
network_perceptron_original = create_author_network(articles_perceptron) 
print(f"{network_perceptron_original.number_of_nodes()=:,}")
print(f"{network_perceptron_original.number_of_edges()=:,}")

network_perceptron_original.number_of_nodes()=4,263
network_perceptron_original.number_of_edges()=12,665


Prune author-based network

In [41]:
# Note: The old notebook wrapped this in tqdm(..., items()) inside the function,
# and the new notebook also has tqdm inside the function.
# We pass 'articles_perceptron' as the records list.
authors_twins_dict = generate_twins_dict(network_perceptron_original, articles_perceptron)
network_perceptron_pruned = prune_network(network_perceptron_original, authors_twins_dict)
print(f"{network_perceptron_pruned.number_of_nodes()=:,}")
print(f"{network_perceptron_pruned.number_of_edges()=:,}")

network_perceptron_pruned_lcc = produce_lcc(network_perceptron_pruned)
print(f"{network_perceptron_pruned_lcc.number_of_nodes()=:,}")
print(f"{network_perceptron_pruned_lcc.number_of_edges()=:,}")

network_perceptron_final = remove_self_loops(network_perceptron_pruned_lcc)
print(f"{network_perceptron_final.number_of_nodes()=:,}")
print(f"{network_perceptron_final.number_of_edges()=:,}")

100%|██████████| 3527/3527 [00:00<00:00, 95476.44it/s]


network_perceptron_pruned.number_of_nodes()=1,507
network_perceptron_pruned.number_of_edges()=3,320
network_perceptron_pruned_lcc.number_of_nodes()=801
network_perceptron_pruned_lcc.number_of_edges()=3,224
network_perceptron_final.number_of_nodes()=801
network_perceptron_final.number_of_edges()=2,983


Save networks

In [42]:
# with open('data/perceptron_works.pkl', 'wb') as f:
#     dill.dump(works_perceptron, f)

# with open('data/perceptron_original.pkl', 'wb') as f:
#     dill.dump(network_perceptron_original, f)
    
with open('perceptron_final_dill.pkl', 'wb') as f:
    dill.dump(network_perceptron_final, f)

# Save the object to a file
with open('perceptron_final.pkl', 'wb') as f:
    pickle.dump(network_perceptron_final, f)

# Save as JSON
data_perceptron = json_graph.node_link_data(network_perceptron_final)
with open('perceptron_final.json', 'w') as f:
    json.dump(data_perceptron, f)

/Users/ignacio/Documents/VS Code/GitHub Repositories/e_network_inequality/.venv/lib/python3.10/site-packages/networkx/readwrite/json_graph/node_link.py:142: FutureWarning: 
The default value will be `edges="edges" in NetworkX 3.6.

To make this warning go away, explicitly set the edges kwarg, e.g.:

  nx.node_link_data(G, edges="links") to preserve current behavior, or
  nx.node_link_data(G, edges="edges") for forward compatibility.
  warnings.warn(
